# Study: Contact Surface Green's Function

This notebook studies the retarded surface Green's function $g^R$ of a semi-infinite contact and links a toy model to the open-boundary recursion used in `quatrex`.

## 1) Scalar contact model

For a nearest-neighbor semi-infinite chain with onsite energy $\varepsilon_0$ and hopping $t$,

$$g^R = [z - t^2 g^R]^{-1}, \quad z = E + i\eta - \varepsilon_0.$$

This gives

$$t^2(g^R)^2 - z g^R + 1 = 0.$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
def g_surface_analytic(E, eps0=0.0, t=1.0, eta=1e-6):
    z = E + 1j * eta - eps0
    disc = np.sqrt(z**2 - 4 * t**2)
    g1 = (z + disc) / (2 * t**2)
    g2 = (z - disc) / (2 * t**2)
    return np.where(np.imag(g1) <= 0, g1, g2)


In [ ]:
eps0 = 0.0
t = 1.0
eta = 1e-6
E = np.linspace(-3.0, 3.0, 1200)

g = g_surface_analytic(E, eps0=eps0, t=t, eta=eta)
z = E + 1j * eta - eps0
quad_residual = t**2 * g**2 - z * g + 1
print(f"Max quadratic residual: {np.max(np.abs(quad_residual)):.3e}")
print(f"Max Im(g^R): {np.max(np.imag(g)):.3e} (retarded branch should be <= 0)")


In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(8, 9), sharex=True)
ax[0].plot(E, np.real(g))
ax[0].set_ylabel("Re(g^R)")

ax[1].plot(E, np.imag(g))
ax[1].set_ylabel("Im(g^R)")

dos = -np.imag(g) / np.pi
ax[2].plot(E, dos)
ax[2].set_ylabel("surface DOS")
ax[2].set_xlabel("Energy E")

for a in ax:
    a.axvline(-2 * t + eps0, color="k", lw=0.8, alpha=0.4)
    a.axvline(2 * t + eps0, color="k", lw=0.8, alpha=0.4)
    a.grid(True, alpha=0.25)

fig.suptitle("Surface Green's function and DOS for a semi-infinite 1D contact")
plt.tight_layout()
plt.show()


## 2) Matrix recursion used in `quatrex`

`quatrex` solves

$$x_{ii} = (a_{ii} - a_{ji} x_{ii} a_{ij})^{-1}.$$

The cell below applies this to a simple 2x2 contact block and checks convergence.

In [ ]:
def matrix_fixed_point(a_ji, a_ii, a_ij, niter=1200, tol=1e-12):
    x = np.linalg.inv(a_ii)
    for it in range(niter):
        x_new = np.linalg.inv(a_ii - a_ji @ x @ a_ij)
        if np.linalg.norm(x_new - x) < tol:
            return x_new, it + 1
        x = x_new
    return x, niter

E0 = 0.3
eta0 = 1e-2
eps = np.array([[0.0, 0.05], [0.05, 0.2]])
coupling = np.array([[-0.3, 0.0], [0.0, -0.24]])

a_ii = (E0 + 1j * eta0) * np.eye(2) - eps
a_ji = coupling
a_ij = coupling.T

x, iters = matrix_fixed_point(a_ji, a_ii, a_ij)
residual = np.linalg.norm(x - np.linalg.inv(a_ii - a_ji @ x @ a_ij))
print("Surface Green function estimate x_ii:")
print(x)
print(f"Converged in {iters} iterations")
print(f"Fixed-point residual norm: {residual:.3e}")


## 3) Optional comparison with `qttools` Sancho-Rubio

If `qttools` is importable from the notebook environment, compare with `SanchoRubio`.

In [ ]:
try:
    from qttools.boundary_conditions.obc.sancho_rubio import SanchoRubio

    sancho = SanchoRubio(max_iterations=400, convergence_tol=1e-12)
    x_sr = sancho((a_ji, a_ii, a_ij), contact="left")
    print("||x_sr - x_fixed_point|| =", np.linalg.norm(x_sr - x))
except Exception as exc:
    print("SanchoRubio comparison skipped:", exc)


## 4) Realistic CNT Hamiltonian (Wannier90 example)

This section uses the realistic carbon-nanotube unit-cell Hamiltonian at repository-relative path:
`examples/w90/carbon-nanotube/inputs-unit-cell/hamiltonian.h5`.

The notebook searches for this file by walking up from the current working directory until it finds the repository root, then loads nearest-cell couplings and computes the contact surface Green's function with `SanchoRubio`.

In [ ]:
import ast
from pathlib import Path

cnt_ready = False

try:
    from qttools.utils.hdf5_utils import load_hdf5_dict
    from qttools.boundary_conditions.obc.sancho_rubio import SanchoRubio
except Exception as exc:
    print('CNT section skipped: qttools unavailable:', exc)
else:
    cnt_h_path = None
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / 'examples/w90/carbon-nanotube/inputs-unit-cell/hamiltonian.h5'
        if candidate.exists():
            cnt_h_path = candidate
            break

    if cnt_h_path is None:
        print('CNT section skipped: could not find examples/w90/carbon-nanotube/inputs-unit-cell/hamiltonian.h5 from current working directory.')
    else:
        H_raw = load_hdf5_dict(str(cnt_h_path))
        H = {tuple(ast.literal_eval(k)): np.asarray(v, dtype=np.complex128) for k, v in H_raw.items()}

        needed = [(0, 0, -1), (0, 0, 0), (0, 0, 1)]
        missing = [k for k in needed if k not in H]
        if missing:
            print(f'CNT section skipped: missing required CNT hoppings: {missing}')
        else:
            H_m1, H_0, H_p1 = H[(0, 0, -1)], H[(0, 0, 0)], H[(0, 0, 1)]
            cnt_ready = True
            print('CNT Hamiltonian path:', cnt_h_path)
            print('CNT Hamiltonian shape:', H_0.shape)
            print('Number of hopping blocks:', len(H))
            print('||H(+1)-H(-1)^†||_F =', np.linalg.norm(H_p1 - H_m1.conj().T))


In [ ]:
if not cnt_ready:
    print('CNT Sancho-Rubio sweep skipped.')
else:
    sancho_cnt = SanchoRubio(max_iterations=400, convergence_tol=1e-10)
    eta_cnt = 1e-6
    E_cnt = np.array([-6.5, -5.5, -4.5, -3.8, -3.0, -2.0, -1.0])

    rows = []
    for e in E_cnt:
        a_ii_cnt = (e + 1j * eta_cnt) * np.eye(H_0.shape[0]) - H_0
        a_ji_cnt = -H_m1
        a_ij_cnt = -H_p1

        g_cnt = sancho_cnt((a_ji_cnt, a_ii_cnt, a_ij_cnt), contact='left')
        resid = np.linalg.norm(g_cnt - np.linalg.inv(a_ii_cnt - a_ji_cnt @ g_cnt @ a_ij_cnt))
        max_im_eig = np.max(np.imag(np.linalg.eigvals(g_cnt)))
        dos_trace = (-np.trace(np.imag(g_cnt)) / np.pi).real
        rows.append((e, resid, max_im_eig, dos_trace))

    print('E(eV)   ||residual||_F    max(Im eig(gR))    -Tr(Im gR)/pi')
    for e, resid, max_im_eig, dos_trace in rows:
        print(f'{e:5.2f}   {resid:13.3e}   {max_im_eig:16.3e}   {dos_trace:14.6f}')


In [ ]:
if not cnt_ready:
    print('CNT diagnostics skipped.')
else:
    e_mid = -3.8
    a_ii_mid = (e_mid + 1j * eta_cnt) * np.eye(H_0.shape[0]) - H_0
    g_mid = sancho_cnt((-H_m1, a_ii_mid, -H_p1), contact='left')
    print('Sample CNT diagnostics at E = -3.8 eV')
    print('gR[0,0] =', g_mid[0, 0])
    print('gR[0,1] =', g_mid[0, 1])
    print('||gR||_F =', np.linalg.norm(g_mid))


## 5) Dense CNT DOS plot

Compute a dense energy sweep and plot energy vs contact DOS for the realistic CNT contact.

In [ ]:
if not cnt_ready:
    print("Dense CNT DOS plot skipped.")
else:
    E_dense = np.linspace(-6.5, -1.0, 220)
    dos_dense = np.empty_like(E_dense)
    eta_dense = 1e-6
    sancho_dense = SanchoRubio(max_iterations=300, convergence_tol=1e-10)

    for i, e in enumerate(E_dense):
        a_ii = (e + 1j * eta_dense) * np.eye(H_0.shape[0]) - H_0
        g_e = sancho_dense((-H_m1, a_ii, -H_p1), contact="left")
        dos_dense[i] = (-np.trace(np.imag(g_e)) / np.pi).real

    plt.figure(figsize=(8, 4.5))
    plt.plot(E_dense, dos_dense, lw=1.8)
    plt.xlabel("Energy E (eV)")
    plt.ylabel(r"DOS proxy $-\mathrm{Tr}(\Im g^R)/\pi$")
    plt.title("CNT contact surface DOS vs energy (dense grid)")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    print(f"Dense grid points: {len(E_dense)}")
    print(f"DOS range: [{dos_dense.min():.6f}, {dos_dense.max():.6f}]")
